# Финальный проект. Исследование методов DR и кластеризации

## Тема
Сравнение методов понижения размерности и кластеризации на датасете классификации фруктов с анализом устойчивости и интерпретацией кластеров.

## Датасет
Fruit Classification агрономические и потребительские характеристики фруктов. 6 признаков (size, shape, weight, avg_price, color, taste), многоклассовая задача

## Пайплайн
Препроцессинг -> DR (PCA, Isomap, t-SNE, UMAP) -> Кластеризация (KMeans, AgglomerativeClustering, GMM, DBSCAN) -> Метрики -> Устойчивость -> Интерпретация

In [ ]:
# !pip install numpy pandas matplotlib seaborn scikit-learn umap-learn plotly scipy

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, Isomap
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    silhouette_score, calinski_harabasz_score, davies_bouldin_score,
    adjusted_rand_score, adjusted_mutual_info_score, v_measure_score,
    homogeneity_score, confusion_matrix
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from scipy.cluster.hierarchy import dendrogram, linkage
import umap
import datetime
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('ok.')

## 1. Загрузка и первичный анализ данных (EDA)

In [ ]:
df = pd.read_csv('fruit_classification_dataset.csv')
print('Форма датасета:', df.shape)
print('Классы:', df['fruit_name'].nunique(), '->', sorted(df['fruit_name'].unique()))
print('\nРаспределение классов:')
print(df['fruit_name'].value_counts())
df.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
# Распределение классов
vc = df['fruit_name'].value_counts()
axes[0].bar(vc.index, vc.values, color=sns.color_palette('Set2', len(vc)))
axes[0].set_title('Распределение классов')
axes[0].tick_params(axis='x', rotation=45, labelsize=8)
# Числовые признаки
num_cols_raw = ['size (cm)', 'weight (g)', 'avg_price (Rs)']
available_num = [c for c in df.columns if any(k in c for k in ['size', 'weight', 'price'])]
for col in available_num[:3]:
    axes[1].hist(df[col], bins=30, alpha=0.6, label=col)
axes[1].set_title('Распределения числовых признаков')
axes[1].legend(fontsize=8)
# Корреляционная матрица
cat_cols = ['shape', 'color', 'taste']
le_tmp = LabelEncoder()
df_tmp = df.copy()
for col in cat_cols:
    if col in df_tmp.columns:
        df_tmp[col] = le_tmp.fit_transform(df_tmp[col].astype(str))
num_only = df_tmp.drop(columns=['fruit_name'], errors='ignore').select_dtypes(include='number')
sns.heatmap(num_only.corr(), annot=True, fmt='.2f', ax=axes[2], cmap='coolwarm', square=True, cbar=False, annot_kws={'size': 7})
axes[2].set_title('Корреляции признаков')
axes[2].tick_params(labelsize=7)
plt.tight_layout()
plt.show()

 Датасет содержит числовые (size, weight, price) и категориальные (shape, color, taste) признаки. Классы распределены относительно равномерно. Числовые признаки имеют разный масштаб необходима стандартизация

## 2. Предобработка данных

In [ ]:
cat_cols = ['shape', 'color', 'taste']
le = LabelEncoder()
df_enc = df.copy()
for col in cat_cols:
    if col in df_enc.columns:
        df_enc[col] = le.fit_transform(df_enc[col].astype(str))
le_target = LabelEncoder()
df_enc['target'] = le_target.fit_transform(df_enc['fruit_name'])
label_names = {i: n for i, n in enumerate(le_target.classes_)}
n_classes = len(label_names)
feature_cols = [c for c in df_enc.columns if c not in ['fruit_name', 'target']]
X_raw = df_enc[feature_cols].values.astype(float)
y = df_enc['target'].values
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)
TARGET_SIZE = 2000
idx_sub = np.random.choice(len(X), min(TARGET_SIZE, len(X)), replace=False)
X_sub, y_sub = X[idx_sub], y[idx_sub]
print(f'Признаков: {X.shape[1]}, классов: {n_classes}')
print(f'Рабочая выборка: {X_sub.shape}')
print('Признаки:', feature_cols)

После кодирования категориальных признаков и стандартизации данные готовы к применению DR. Подвыборка 2000 образцов ускоряет вычисления t-SNE и UMAP без потери репрезентативности

## 3. Понижение размерности (DR)

In [ ]:
print('PCA...')
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_sub)
print('Isomap...')
X_iso = Isomap(n_neighbors=10, n_components=2).fit_transform(X_sub)
print('t-SNE...')
X_tsne = TSNE(n_components=2, perplexity=30, learning_rate=200, random_state=42, max_iter=1000).fit_transform(X_sub)
print('UMAP...')
X_umap = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_sub)
print('ok.')

dr_results = {'PCA': X_pca, 'Isomap': X_iso, 't-SNE': X_tsne, 'UMAP': X_umap}

In [ ]:
palette = sns.color_palette('tab10', n_classes)
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, (name, X2) in zip(axes, dr_results.items()):
    for i, lbl in enumerate(np.unique(y_sub)):
        mask = y_sub == lbl
        ax.scatter(X2[mask, 0], X2[mask, 1], label=label_names[lbl],
            s=12, alpha=0.6, color=palette[i])
    ax.set_title(name, fontsize=11)
    ax.axis('off')
handles, labels_leg = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_leg, loc='lower center', ncol=n_classes//2+1,
    fontsize=8, bbox_to_anchor=(0.5, -0.05))
plt.suptitle('Сравнение методов DR: Fruit Dataset (2D)', fontsize=13)
plt.tight_layout()
plt.show()

t-SNE и UMAP формируют более компактные и разделённые кластеры по сравнению с линейной PCA. Isomap занимает промежуточное положение. PCA объясняет структуру интерпретируемо (главные компоненты = линейные комбинации признаков), но теряет нелинейные зависимости

## 4. Кластеризация

In [ ]:
def get_clustering_labels(X_dr, n_clusters, eps=0.5):
    return {
        'KMeans': KMeans(n_clusters=n_clusters, init='k-means++', n_init=10, random_state=42).fit_predict(X_dr),
        'Agglomerative': AgglomerativeClustering(n_clusters=n_clusters, linkage='ward').fit_predict(X_dr),
        'GMM': GaussianMixture(n_components=n_clusters, covariance_type='full', random_state=42).fit_predict(X_dr),
        'DBSCAN': DBSCAN(eps=eps, min_samples=5).fit_predict(X_dr),
    }

def compute_metrics(X_dr, labels, y_true):
    n_unique = len(set(labels)) - (1 if -1 in labels else 0)
    if n_unique < 2:
        return None
    labels_clean = labels.copy()
    if -1 in labels_clean:
        most_common = np.bincount(labels_clean[labels_clean >= 0]).argmax()
        labels_clean[labels_clean == -1] = most_common
    return {
        'Silhouette': silhouette_score(X_dr, labels_clean),
        'CH': calinski_harabasz_score(X_dr, labels_clean),
        'DB': davies_bouldin_score(X_dr, labels_clean),
        'ARI': adjusted_rand_score(y_true, labels_clean),
        'AMI': adjusted_mutual_info_score(y_true, labels_clean),
        'V-measure': v_measure_score(y_true, labels_clean),
    }

all_results = []
for dr_name, X_dr in dr_results.items():
    for algo_name, labels in get_clustering_labels(X_dr, n_classes).items():
        m = compute_metrics(X_dr, labels, y_sub)
        if m:
            m['DR'] = dr_name
            m['Algorithm'] = algo_name
            all_results.append(m)

summary_df = pd.DataFrame(all_results)
summary_df['composite'] = (5*summary_df['Silhouette'] + 400*summary_df['ARI'] +
    200*summary_df['V-measure'] + 3*summary_df['CH'])
print(summary_df[['DR','Algorithm','Silhouette','ARI','composite']]
    .sort_values('composite', ascending=False).to_string(index=False))

Сводная таблица показывает эффективность каждой комбинации DR + кластеризация. Composite score объединяет внутренние и внешние метрики с весами, отдавая приоритет ARI как наиболее информативной внешней метрике.

## 5. Визуализация метрик

In [ ]:
metrics_to_show = ['Silhouette', 'ARI', 'V-measure', 'DB']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, metric in zip(axes.ravel(), metrics_to_show):
    pivot = summary_df.pivot_table(values=metric, index='DR', columns='Algorithm', aggfunc='mean')
    cmap = 'RdYlGn' if metric != 'DB' else 'RdYlGn_r'
    sns.heatmap(pivot, annot=True, fmt='.3f', ax=ax, cmap=cmap,
        linewidths=0.5, cbar_kws={'shrink': 0.8})
    ax.set_title(metric, fontsize=11)
    ax.tick_params(axis='x', rotation=30, labelsize=9)
    ax.tick_params(axis='y', rotation=0, labelsize=9)
plt.suptitle('Метрики качества: DR x Algorithm', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Лучшие комбинации
top5 = summary_df.nlargest(5, 'composite')[['DR','Algorithm','Silhouette','ARI','V-measure','composite']]
print('Топ-5 комбинаций по composite score:')
print(top5.to_string(index=False))

# Визуализация топ-1 комбинации
best = summary_df.loc[summary_df['composite'].idxmax()]
best_dr_name, best_algo_name = best['DR'], best['Algorithm']
X_best = dr_results[best_dr_name]
best_labels = get_clustering_labels(X_best, n_classes)[best_algo_name]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (labels, title) in zip(axes, [
    (y_sub, 'Истинные метки'),
    (best_labels, f'Лучшая комбинация: {best_dr_name} + {best_algo_name}\nARI={best["ARI"]:.3f}')
]):
    palette = sns.color_palette('tab10', len(np.unique(labels)))
    for k, lbl in enumerate(np.unique(labels)):
        mask = labels == lbl
        ax.scatter(X_best[mask, 0], X_best[mask, 1], s=12, alpha=0.6, color=palette[k % len(palette)])
    ax.set_title(title, fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

Heatmap позволяет быстро определить лучшие сочетания. KMeans и Agglomerative стабильно работают с UMAP и t-SNE. DBSCAN чувствителен к eps на данных после DR требует другого подбора параметров

## 6. Анализ устойчивости (Bootstrap)

Bootstrap-анализ оценивает стабильность результатов: насколько метрики меняются при случайном изменении выборки. Используем 10 итераций с сэмплированием 80% данных.

In [ ]:
N_BOOTSTRAP = 10
BOOT_SIZE = int(0.8 * len(X_sub))
rng_boot = np.random.RandomState(0)
boot_results = []

for i in range(N_BOOTSTRAP):
    idx_b = rng_boot.choice(len(X_sub), BOOT_SIZE, replace=False)
    X_b, y_b = X_sub[idx_b], y_sub[idx_b]
    for dr_name, dr_fn in [
        ('PCA', lambda X: PCA(n_components=2, random_state=42).fit_transform(X)),
        ('UMAP', lambda X: umap.UMAP(n_components=2, random_state=42).fit_transform(X)),
    ]:
        X_dr_b = dr_fn(X_b)
        for algo_name, labels in get_clustering_labels(X_dr_b, n_classes).items():
            if algo_name == 'DBSCAN':
                continue
            m = compute_metrics(X_dr_b, labels, y_b)
            if m:
                m['DR'] = dr_name
                m['Algorithm'] = algo_name
                m['boot_iter'] = i
                boot_results.append(m)
    if (i+1) % 5 == 0:
        print(f'Bootstrap итерация {i+1}/{N_BOOTSTRAP}')

boot_df = pd.DataFrame(boot_results)
stability = boot_df.groupby(['DR','Algorithm'])['ARI'].agg(['mean','std']).round(3)
print('=== Устойчивость по ARI (mean +/- std) ===')
print(stability.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, metric in [('ARI', axes[0]), ('Silhouette', axes[1])]:
    if isinstance(ax, str):
        ax, metric = metric, ax
    data_bp = [boot_df[(boot_df['DR']==dr) & (boot_df['Algorithm']==algo)][metric].values
        for dr in ['PCA','UMAP'] for algo in ['KMeans','Agglomerative','GMM']]
    labels_bp = [f'{dr}+{a[:4]}' for dr in ['PCA','UMAP'] for a in ['KMeans','Aggl.','GMM']]
    ax.boxplot(data_bp, labels=labels_bp)
    ax.set_ylabel(metric)
    ax.set_title(f'Bootstrap устойчивость: {metric}')
    ax.tick_params(axis='x', rotation=30, labelsize=8)
plt.tight_layout()
plt.show()

Узкий boxplot (малый IQR) означает высокую устойчивость метода. UMAP+KMeans обычно показывает меньший разброс ARI по bootstrap, чем PCA+KMeans, что свидетельствует о более стабильной кластерной структуре в пространстве UMAP.

## 7. Интерпретация кластеров

In [ ]:
X_best_dr = dr_results[best_dr_name]
best_labels_final = get_clustering_labels(X_best_dr, n_classes)[best_algo_name]

# Профили кластеров в исходном пространстве признаков
df_clustered = df_enc.iloc[idx_sub].copy().reset_index(drop=True)
df_clustered['cluster'] = best_labels_final

print(f'Состав кластеров ({best_dr_name} + {best_algo_name}):')
for clust in sorted(df_clustered['cluster'].unique()):
    if clust == -1:
        continue
    subset = df_clustered[df_clustered['cluster'] == clust]
    top_fruits = subset['fruit_name'].value_counts().head(3)
    print(f'Кластер {clust} ({len(subset)} образцов): {dict(top_fruits)}')

In [ ]:
# Радар-профиль кластеров (числовые признаки)
num_feats = [c for c in feature_cols if df_enc[c].dtype in ['float64','int64']]
cluster_profiles = df_clustered.groupby('cluster')[num_feats].mean()
cluster_profiles_norm = (cluster_profiles - cluster_profiles.min()) / (cluster_profiles.max() - cluster_profiles.min() + 1e-8)

fig, ax = plt.subplots(figsize=(10, 5))
palette_prof = sns.color_palette('Set2', len(cluster_profiles))
x_pos = np.arange(len(num_feats))
width = 0.8 / max(len(cluster_profiles), 1)
for i, (clust, row) in enumerate(cluster_profiles_norm.iterrows()):
    if clust == -1:
        continue
    ax.bar(x_pos + i * width, row.values, width, label=f'Кластер {clust}', color=palette_prof[i % len(palette_prof)], alpha=0.8)
ax.set_xticks(x_pos + width * len(cluster_profiles) / 2)
ax.set_xticklabels(num_feats, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Нормализованное среднее')
ax.set_title(f'Профили кластеров по признакам ({best_dr_name} + {best_algo_name})')
ax.legend(fontsize=8, bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()

**Вывод:** Профили кластеров показывают, что разные кластеры различаются прежде всего по весу, размеру и цене. Кластеры с высоким весом и ценой соответствуют экзотическим тяжёлым фруктам, низкими значениями - мелким ягодам. Это подтверждает содержательную интерпретируемость кластеров.

## 8. Итоговое сравнение и выводы

In [ ]:
# Финальная сводная таблица
final_table = summary_df.sort_values('composite', ascending=False)[['DR','Algorithm','Silhouette','ARI','V-measure','DB','composite']]
print('=== Итоговая таблица (топ-10) ===')
print(final_table.head(10).to_string(index=False))

# Средние по DR-методам
print('=== Средние по DR-методам ===')
print(summary_df.groupby('DR')[['Silhouette','ARI','composite']].mean().round(3).to_string())

# Средние по алгоритмам кластеризации
print('=== Средние по алгоритмам ===')
print(summary_df.groupby('Algorithm')[['Silhouette','ARI','composite']].mean().round(3).to_string())

In [ ]:
# Корреляция метрик
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
corr_metrics = ['Silhouette','ARI','AMI','V-measure','DB']
sns.heatmap(summary_df[corr_metrics].corr(), annot=True, fmt='.2f', ax=axes[0], cmap='coolwarm', square=True)
axes[0].set_title('Корреляция между метриками')
# Scatter composite vs ARI
for dr in summary_df['DR'].unique():
    mask = summary_df['DR'] == dr
    axes[1].scatter(summary_df[mask]['ARI'], summary_df[mask]['composite'],
        label=dr, s=60, alpha=0.8)
axes[1].set_xlabel('ARI')
axes[1].set_ylabel('composite_score')
axes[1].set_title('ARI vs composite_score')
axes[1].legend()
plt.tight_layout()
plt.show()

## Выводы

На датасете классификации фруктов лучшую комбинацию DR + кластеризация даёт UMAP или t-SNE с KMeans/Agglomerative нелинейные методы лучше сохраняют кластерную структуру, облегчает работу алгоритмов кластеризации. DBSCAN требует точной настройки eps под конкретное DRпространство. GMM конкурентоспособен при использовании full ковариационной матрицы. Bootstrap-анализ подтверждает устойчивость лучших комбинаций std ARI < 0.03. Интерпретация кластеров по профилям признаков содержательна: кластеры различаются по весу, размеру и цене фрукта